In [73]:
from collections import defaultdict, Counter
import numpy as np
import random
import math

We test bigram and trigram approaches for next word prediction in the Penn-Treebank dataset, which contains snippets from the Wall Street Journal. In this data set, there contains `<unk>` which denotes an unknown word.

In [22]:
bigram = defaultdict(Counter)
trigram = defaultdict(Counter)

In [23]:
with open("../data/penn-treebank/ptb.train.txt") as f:
    for line in f:
        words = line.strip().split()
        tokens = ["<s>", "<s>"] + words + ["</s>"]

        for i in range(len(tokens)):
            if i >= 1: bigram[tokens[i-1]][tokens[i]] += 1
            if i>= 2: trigram[(tokens[i-2], tokens[i-1])][tokens[i]] += 1

The tokens `<s>` and `</s>` are used to denote the start and end of a sentence, delimited by a newline in the data. Two `<s>` tokens are required for a trigram, which interprets the third word as the first word of a sentence with two prior words being the start of the sentence.

In [45]:
def sample_bigram(w1):
    counts = bigram[w1]

    if not counts: return "</s>"

    words = list(counts.keys())
    weights = list(counts.values())

    # return a random choice weighted by the number of times it showed up in the training set
    return random.choices(words, weights=weights, k=1)[0]

def sample_trigram(w1, w2):
    counts = trigram[(w1, w2)]

    if not counts: return "</s>"

    words = list(counts.keys())
    weights = list(counts.values())

    return random.choices(words, weights=weights, k=1)[0]

We can now do something fairly interesting - generate a completely random sentence with bigrams and trigrams. Of course, our sentences are derived from financial news, and so we should expect words relating to financial markets. 

We can do this by starting with `<s>` and `<s> <s>` for bigram and trigram respectively and then sampling from the counters.

In [52]:
word = "<s>"
sentence = []
for i in range(1, 50):
    word = sample_bigram(word)
    sentence.append(word)

    if word == "</s>": break

print(' '.join(sentence))

each year according to $ N to affect prices for a bit closer to N reports from <unk> </s>


In [54]:
word_1 = "<s>"
word_2 = "<s>"
sentence = []
for i in range(1, 50):
    word_temp = sample_trigram(word_1, word_2)

    # exchange
    t = word_2
    word_2 = word_temp
    word_1 = t

    sentence.append(word_temp)

    if word_temp == "</s>": break

print(' '.join(sentence))

revenue climbed N to N and american telephone & telegraph co. was unchanged at $ N billion investment fund with a year selling by more than $ N million up N N to N chevron gained N to the lead slashed north american factory sales of stock funds simply


The above sentences do not really make any human sense. It almost feels as though the bigrams are too greedy and are too short-sighted, whereas the trigrams are a bit too long-sighted (most evident in the doubling of "the"). We could instead blend the probability of a word $w_n$ for the previous two words $w_{n-1}$ and $w_{n-2}$ by

$$ P(w_n | w_{n-1}, w_{n-2}) = \lambda P_{bigram}(w_n|w_{n-1}) + (1 - \lambda)P_{trigram}(w_n|w_{n-1}, w_{n-2})$$

In [56]:
def sample_blended(w1, w2, L=0.5):
    bigram_cands = bigram[w2]
    trigram_cands = trigram[(w1, w2)]

    cands = set(bigram_cands.keys()) | set(trigram_cands.keys())

    # no intersection of candidates
    if not cands: return "</s>"

    bi_total, tri_total = sum(bigram_cands.values()), sum(trigram_cands.values())
    words, weights = [], []

    for word in cands:
        p_bigram = (bigram_cands[word] / bi_total if bi_total > 0 else 0)
        p_trigram = (trigram_cands[word] / tri_total if tri_total > 0 else 0)
        p_blended = L * p_bigram + (1 - L) * p_trigram

        words.append(word)
        weights.append(p_blended)

    return random.choices(words, weights=weights, k=1)[0]

Consider the below.

In [62]:
word_1 = "<s>"
word_2 = "<s>"
sentence = []

for _ in range(50):
    word_temp = sample_blended(word_1, word_2)

    if word_temp == "</s>":
        break

    sentence.append(word_temp)

    word_1, word_2 = word_2, word_temp

print(" ".join(sentence))

futures contract in the dow jones medical industries plc won overwhelming shareholder approval of the cultural and beauty products to control lin 's victories the explosions <unk> research says jeffrey coors N million of pollution control </s>


And the results are... much more understandable? Of course, it is not all intelligble and definitely not all valid english but some parts of it are somewhat readable. We can find the log loss of our blended model by defining
$$
\begin{align*}
    L_n &= -\log P(w_n | w_{n-1}, w_{n-2}) \\
    &= -\log \left[ \lambda P_{bigram}(w_n|w_{n-1}) + (1 - \lambda)P_{trigram}(w_n|w_{n-1}, w_{n-2}) \right]
\end{align*}
$$

In [66]:
def loss_at_n(w1, w2, word, L=0.5):
    bigram_cands = bigram[w2]
    trigram_cands = trigram[(w1, w2)]

    cands = set(bigram_cands.keys()) | set(trigram_cands.keys())

    # no intersection of candidates
    if not cands: return 0

    bi_total, tri_total = sum(bigram_cands.values()), sum(trigram_cands.values())

    p_bigram = (bigram_cands[word] / bi_total if bi_total > 0 else 0)
    p_trigram = (trigram_cands[word] / tri_total if tri_total > 0 else 0)
    return (L * p_bigram + (1 - L) * p_trigram)

In [70]:
total_loss = 0
num_predictions = 0

with open('../data/penn-treebank/ptb.test.txt') as f:
    for lines in f:
        words = line.strip().split()
        tokens = ["<s>", "<s>"] + words + ["</s>"]

        for i in range(2, len(tokens)):
            loss_i = loss_at_n(tokens[i-2], tokens[i-1], tokens[i])

            # this is kinda cheating... but WIP
            if loss_i == 0: next

            total_loss += -math.log(loss_i)
            num_predictions += 1

How do we evaluate the accuracy of a language model? A useful metric is *perplexity*, which gives a feel for how many words at each step the model is "considering", under the assumption that all words are equally likely. Essentially, it shows us how fine-graned the model is.

$$
\begin{align*}

\verb|Perplexity| &= \exp^L \\
&= \exp^{-\frac{1}{N}\sum_{t=1}^N \log P(w_t | \text{context})}

\end{align*}
$$

Another way is considering how often the model is correct. We gather this from the geometric probability. Consider some probabilities of the correct words in a sequence $\mathbf{p} = p_1, p_2, \dots, p_N$. The total probability of the sequence is 

$$ P(\verb|sequence|) = \prod_{t=1}^N p_t$$

A typical probability of getting the word correct could be derived from the geometric mean as the sequence probability is multiplicative of the per-word probability.

$$ \left(\prod_{t=1}^N p_t \right)^{1/N}$$

Through some mathematical niceties, this can actually just be calculated by doing $\exp^{-L}$.

In [80]:
average_loss = total_loss / num_predictions
print(f'Perplexity: {round(np.exp(average_loss), 3)}')
print(f'Geometric mean correctness: {round(np.exp(-average_loss), 3) * 100}%')

Perplexity: 9.372
Geometric mean correctness: 10.7%


At each step the model has ~9.4 words to guess from, which isn't great! The baseline naive model below an $n$-gram model, which would have a much lower mean correctness due to the size of the vocabulary. We could also find the optimal $\lambda$; which gives us how much to weight our model towards the bigram or the trigram. Trigrams have richer context, so it could be intuitive that we give more weight to the trigram, but we test this empirically with varying values of $\lambda$.